# MNIST: Red Densa + Optuna + MLflow + Regularización

Actividad completa para:
1. Buscar una arquitectura densa con Optuna, **sin regularización**.
2. Registrar experimentos con MLflow.
3. Entrenar la mejor arquitectura.
4. Comparar Base, L1, L2, L1-L2, Dropout y Dropout + L1-L2.
5. Generar tablas y gráficas para el reporte.

> Los resultados numéricos se generan al ejecutar el notebook; no están inventados.

## 0. Instalar dependencias

In [ ]:
!pip -q install optuna mlflow dagshub

## 1. Importaciones y configuración

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import optuna
import mlflow
import mlflow.tensorflow

from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)
print("MLflow:", mlflow.__version__)
print("Dispositivos:", tf.config.list_physical_devices())

## 2. Cargar y preparar MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Entrenamiento:", x_train.shape, y_train.shape)
print("Prueba:", x_test.shape, y_test.shape)

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Separación reproducible: 90% train y 10% validation
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(x_train))
n_val = int(0.10 * len(x_train))

val_idx = idx[:n_val]
train_idx = idx[n_val:]

x_val, y_val = x_train[val_idx], y_train[val_idx]
x_train_sub, y_train_sub = x_train[train_idx], y_train[train_idx]

print("Train:", x_train_sub.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

## 3. Visualizar ejemplos

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train_sub[:10], y_train_sub[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Clase: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Constructor de la red

La red es **densa secuencial**, no convolucional. Cada imagen de 28×28 se transforma mediante `Flatten` en 784 entradas.

En esta etapa no se usa ninguna regularización.

In [ ]:
def build_dense_model(
    n_layers,
    units,
    activations,
    optimizer_name,
    learning_rate,
    regularization=None,
    dropout_rate=0.0
):
    model = keras.Sequential(name="MNIST_Dense")
    model.add(layers.Input(shape=(28, 28)))
    model.add(layers.Flatten())

    for i in range(n_layers):
        reg = None
        if regularization == "l1":
            reg = regularizers.l1(1e-4)
        elif regularization == "l2":
            reg = regularizers.l2(1e-4)
        elif regularization == "l1_l2":
            reg = regularizers.l1_l2(l1=1e-4, l2=1e-4)

        model.add(layers.Dense(
            units[i],
            activation=activations[i],
            kernel_regularizer=reg
        ))

        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation="softmax"))

    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    elif optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        raise ValueError("Optimizador no reconocido")

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## 5. Configurar MLflow

Para cumplir con el requisito de **entregar un enlace del servidor donde se puedan ver las gráficas**, se registra en **DagsHub** (servidor MLflow remoto conectado a tu repositorio de GitHub).

- **Opción A (recomendada):** DagsHub → genera un enlace público.
- **Opción B:** MLflow local (`./mlruns`) → sin enlace público (solo para pruebas).

Rellena `DAGSHUB_USER`, `DAGSHUB_REPO` y tu **token** (DagsHub → *Settings → Tokens*).

In [ ]:
# ============================================================
# Configuracion de MLflow
# ------------------------------------------------------------
# Opcion A (RECOMENDADA para la entrega): DagsHub -> MLflow remoto.
#   Genera un enlace PUBLICO donde el profesor puede ver las graficas.
#   Requiere: cuenta en https://dagshub.com, un repositorio creado alli
#   y tu token de acceso (DagsHub -> Settings -> Tokens).
#
# Opcion B: MLflow local (file:./mlruns). No genera enlace publico.
# ============================================================

USAR_DAGSHUB = True   # cambialo a False si solo quieres registro local

# --- Rellena con tus datos de DagsHub ---
DAGSHUB_USER  = "TU_USUARIO_DAGSHUB"              # p.ej. brendamones29
DAGSHUB_REPO  = "Redes_Neuronales_Artificiales"  # nombre del repo en DagsHub
DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN", "")  # pega tu token aqui o exportalo

if USAR_DAGSHUB and DAGSHUB_USER != "TU_USUARIO_DAGSHUB":
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
    print("MLflow -> DagsHub (remoto)")
    print("ENLACE DEL SERVIDOR (para el reporte):")
    print(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}/experiments")
else:
    MLFLOW_DIR = "./mlruns"
    mlflow.set_tracking_uri(f"file:{MLFLOW_DIR}")
    print("MLflow -> local:", os.path.abspath(MLFLOW_DIR))
    print("(Sin enlace publico: configura DagsHub para la entrega.)")

mlflow.set_experiment("MNIST_Dense_Optuna")
print("Tracking URI:", mlflow.get_tracking_uri())

## 6. Función objetivo de Optuna

Se exploran:
- 1–4 capas
- 32–512 neuronas
- ReLU, tanh, ELU
- Adam, RMSprop, SGD
- learning rate entre 1e-4 y 1e-2
- batch size 32, 64 o 128

No se utiliza regularización.

In [ ]:
N_TRIALS = 30
EPOCHS_OPTUNA = 10

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)

    units = [
        trial.suggest_categorical(
            f"units_{i}", [32, 64, 128, 256, 512]
        )
        for i in range(n_layers)
    ]

    activations = [
        trial.suggest_categorical(
            f"activation_{i}", ["relu", "tanh", "elu"]
        )
        for i in range(n_layers)
    ]

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["adam", "rmsprop", "sgd"]
    )

    learning_rate = trial.suggest_float(
        "learning_rate", 1e-4, 1e-2, log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size", [32, 64, 128]
    )

    model = build_dense_model(
        n_layers, units, activations,
        optimizer_name, learning_rate
    )

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.log_params(trial.params)

        history = model.fit(
            x_train_sub, y_train_sub,
            validation_data=(x_val, y_val),
            epochs=EPOCHS_OPTUNA,
            batch_size=batch_size,
            verbose=0
        )

        best_val_acc = float(max(history.history["val_accuracy"]))
        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.log_metric("final_val_loss", float(history.history["val_loss"][-1]))

    trial.set_user_attr("best_val_accuracy", best_val_acc)
    keras.backend.clear_session()
    return best_val_acc

## 7. Ejecutar Optuna

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="MNIST_Dense_Search"
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("Mejor validation accuracy:", study.best_value)
print("\nMejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

## 8. Tabla y gráficas de Optuna

In [ ]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "state")
)
display(trials_df.sort_values("value", ascending=False).head(10))

try:
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.show()
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.show()
except Exception as e:
    print("Las gráficas de Optuna no pudieron mostrarse:", e)

## 9. Reconstruir la mejor arquitectura

In [ ]:
best_params = study.best_params
best_n_layers = best_params["n_layers"]

best_units = [best_params[f"units_{i}"] for i in range(best_n_layers)]
best_activations = [best_params[f"activation_{i}"] for i in range(best_n_layers)]
best_optimizer = best_params["optimizer"]
best_lr = best_params["learning_rate"]
best_batch_size = best_params["batch_size"]

print("Mejor arquitectura")
print("Capas:", best_n_layers)
print("Neuronas:", best_units)
print("Activaciones:", best_activations)
print("Optimizador:", best_optimizer)
print("Learning rate:", best_lr)
print("Batch size:", best_batch_size)
print("Validation accuracy:", study.best_value)

## 10. Función para entrenar los modelos finales

In [ ]:
EPOCHS_FINAL = 20

def train_model(
    regularization=None,
    dropout_rate=0.0,
    name="model"
):
    keras.backend.clear_session()

    model = build_dense_model(
        n_layers=best_n_layers,
        units=best_units,
        activations=best_activations,
        optimizer_name=best_optimizer,
        learning_rate=best_lr,
        regularization=regularization,
        dropout_rate=dropout_rate
    )

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model_type", name)
        mlflow.log_param("regularization", str(regularization))
        mlflow.log_param("dropout_rate", dropout_rate)

        history = model.fit(
            x_train_sub, y_train_sub,
            validation_data=(x_val, y_val),
            epochs=EPOCHS_FINAL,
            batch_size=best_batch_size,
            verbose=1
        )

        test_loss, test_acc = model.evaluate(
            x_test, y_test, verbose=0
        )

        mlflow.log_metric("test_loss", float(test_loss))
        mlflow.log_metric("test_accuracy", float(test_acc))

    return model, history, test_loss, test_acc

## 11. Entrenar los seis casos

In [ ]:
configs = {
    "Base": (None, 0.0),
    "L1": ("l1", 0.0),
    "L2": ("l2", 0.0),
    "L1-L2": ("l1_l2", 0.0),
    "Dropout": (None, 0.30),
    "Dropout + L1-L2": ("l1_l2", 0.30),
}

results = {}
models = {}

for name, (reg, dropout) in configs.items():
    print(f"\n===== {name} =====")
    model, history, test_loss, test_acc = train_model(
        regularization=reg,
        dropout_rate=dropout,
        name=name
    )
    models[name] = model
    results[name] = {
        "history": history,
        "test_loss": test_loss,
        "test_accuracy": test_acc
    }

print("\nTodos los modelos fueron entrenados.")

## 12. Tabla comparativa

In [ ]:
rows = []

for name, result in results.items():
    h = result["history"].history
    train_acc = max(h["accuracy"])
    val_acc = max(h["val_accuracy"])

    rows.append({
        "Modelo": name,
        "Mejor Train Accuracy": train_acc,
        "Mejor Val Accuracy": val_acc,
        "Test Accuracy": result["test_accuracy"],
        "Test Loss": result["test_loss"],
        "Min Val Loss": min(h["val_loss"]),
        "Brecha Train-Val": train_acc - val_acc
    })

results_df = pd.DataFrame(rows)
display(results_df.sort_values("Test Accuracy", ascending=False).style.format({
    "Mejor Train Accuracy": "{:.4f}",
    "Mejor Val Accuracy": "{:.4f}",
    "Test Accuracy": "{:.4f}",
    "Test Loss": "{:.4f}",
    "Min Val Loss": "{:.4f}",
    "Brecha Train-Val": "{:.4f}"
}))

## 13. Curvas de Validation Accuracy

In [ ]:
plt.figure(figsize=(12, 7))
for name, result in results.items():
    plt.plot(
        result["history"].history["val_accuracy"],
        label=name
    )
plt.xlabel("Época")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy por modelo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 14. Curvas de Validation Loss

In [ ]:
plt.figure(figsize=(12, 7))
for name, result in results.items():
    plt.plot(
        result["history"].history["val_loss"],
        label=name
    )
plt.xlabel("Época")
plt.ylabel("Validation Loss")
plt.title("Validation Loss por modelo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 15. Curvas Train vs Validation del modelo base

In [ ]:
h = results["Base"]["history"].history

plt.figure(figsize=(10, 6))
plt.plot(h["accuracy"], label="Train Accuracy")
plt.plot(h["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Modelo base: Train vs Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(h["loss"], label="Train Loss")
plt.plot(h["val_loss"], label="Validation Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Modelo base: Train vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 16. Comparación de Test Accuracy

Esta gráfica permite comparar el desempeño sobre el conjunto de prueba.

In [ ]:
plot_df = results_df.sort_values("Test Accuracy")

plt.figure(figsize=(10, 6))
plt.barh(plot_df["Modelo"], plot_df["Test Accuracy"])
plt.xlabel("Test Accuracy")
plt.ylabel("Modelo")
plt.title("Test Accuracy")
plt.xlim(
    max(0, plot_df["Test Accuracy"].min() - 0.02),
    min(1, plot_df["Test Accuracy"].max() + 0.01)
)
plt.grid(axis="x", alpha=0.3)
plt.show()

## 17. Análisis de sobreajuste

La brecha `Train Accuracy - Validation Accuracy` sirve como indicador descriptivo.

No debe utilizarse sola para afirmar que existe sobreajuste: también se deben revisar las curvas de `loss`, `accuracy`, validation accuracy y el desempeño en test.

In [ ]:
analysis_df = results_df.copy()
display(analysis_df[
    ["Modelo", "Mejor Train Accuracy", "Mejor Val Accuracy",
     "Test Accuracy", "Brecha Train-Val"]
].sort_values("Brecha Train-Val").style.format({
    "Mejor Train Accuracy": "{:.4f}",
    "Mejor Val Accuracy": "{:.4f}",
    "Test Accuracy": "{:.4f}",
    "Brecha Train-Val": "{:.4f}"
}))

## 18. Exportar resultados

In [ ]:
results_df.to_csv("resultados_regularizacion.csv", index=False)
study.trials_dataframe().to_csv("optuna_trials.csv", index=False)

os.makedirs("modelos_mnist", exist_ok=True)

for name, model in models.items():
    safe = name.lower().replace(" ", "_").replace("+", "plus")
    model.save(f"modelos_mnist/{safe}.keras")

print("Archivos generados:")
print("- resultados_regularizacion.csv")
print("- optuna_trials.csv")
print("- modelos_mnist/")

# 19. Texto guía para el reporte

Después de ejecutar el notebook, completa la conclusión con los resultados reales.

### Optuna
Describe:
- número de trials;
- hiperparámetros explorados;
- mejor arquitectura;
- validation accuracy obtenida.

### Regularización
Para cada caso (L1, L2, L1-L2, Dropout y Dropout + L1-L2), compara:
- train accuracy;
- validation accuracy;
- test accuracy;
- validation loss;
- brecha train-validation;
- forma de las curvas.

### Pregunta principal
**¿La regularización ayudó a mejorar la eficiencia antes de haber sobreajuste?**

La respuesta debe basarse en tus resultados. Por ejemplo, si una regularización reduce la brecha entre entrenamiento y validación y mantiene o mejora el desempeño de validación/test, puedes argumentar que ayudó a mejorar la generalización. Si el desempeño empeora, debes indicarlo.

No asumas que una técnica es mejor antes de observar los resultados.

## 20. Enlace del servidor MLflow (entrega)

**Opción A – DagsHub (recomendada, genera enlace público):**

Si en la sección 5 configuraste `USAR_DAGSHUB = True` con tus datos, todos los experimentos
(los trials de Optuna y los 6 modelos de regularización) quedan en tu servidor MLflow remoto.

El enlace que debes poner en el reporte es:

```
https://dagshub.com/<TU_USUARIO>/Redes_Neuronales_Artificiales/experiments
```

Ahí el profesor puede ver las gráficas, métricas y comparar runs sin instalar nada.

**Opción B – MLflow local (solo pruebas, sin enlace público):**

Si usaste el modo local, puedes abrir la interfaz con la celda siguiente y, en Colab,
exponer el puerto 5000 con el mecanismo de tunneling que uses. Para la entrega final,
usa DagsHub.

In [ ]:
# Solo para la Opcion B (MLflow local). Con DagsHub NO necesitas esta celda.
!nohup mlflow ui --backend-store-uri ./mlruns --host 0.0.0.0 --port 5000 > mlflow.log 2>&1 &
print("MLflow UI local iniciada en el puerto 5000.")